Given a start text, let's list the next n most probable tokens (with their probabilities).

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

from gptbench import Sample, empty_config

In [ ]:
ben = Sample(seed=0xcabc0ffee)

# set config settings
cfg = empty_config()
cfg.model.set(dtype='bfloat16') # halve the memory requirements

ben.init_pretrained('gpt2', cfg) # 'gpt2' or 'gpt-xl' if your GPU can handle it

In [ ]:
# What are the most probable tokens after 'The sky is'
ben.model_next(10, text='The sky is')

Note that most words were tokenized to start with a space character, hence ' blue' in the returned values.

Returned tuple means: probability, token_text, token_id

Let's build a function to follow and list the top 3 probabilities along n steps/tokens:

In [ ]:
def follow(text, steps):
    for s in range(steps):
        gen = ben.model_next(3, text=text)
        print(f"============================== Step {s+1}")
        print(f"'{text}' -> ", end='')
        for o in (gen):
            print(f"'{o[1]}' ({o[0]*100:.2f}%), ", end='')
        print()

        text += gen[0][1]

follow('The sky is', 5)

Using the probability of the next token we can score which one from a number of options has the highest probability. For example:

'This sky is' ->

- ' blue'

- ' yellow'

By checking which token as the highest probability, we can choose the winner:

In [ ]:
def choose(text, options):
    print(f"Prompt: '{text}'")
    tokens = []
    for o in options:
        enc = ben.train_dataset.encode(o)
        assert len(enc) == 1, f"Only single token options: '{o}' has {len(enc)} tokens"
        tokens.append(enc[0])
        
    probs = ben.model_probs(text=text)

    best_prob = 0
    for i,t in enumerate(tokens):
        prob = probs[t].item()
        option_text = options[i]
        print(f"- '{option_text}': prob={prob*100:.1f}%")
        
        if(prob > best_prob):
            best_prob = prob
            best_text = option_text

    print(f"Best choice: '{best_text}'")
    print('-->', text + best_text)

text = 'The sky is'
# note that we're prefixing with a space character:
options = [' blue', ' yellow']

choose(text, options)

In [ ]:
text = 'The capital of Portugal is'
# note that we're prefixing with a space character:
options = [' Lisbon', ' Cuba', ' Paris', ' white']

choose(text, options)

In [ ]:
text = '1+1='
# note that we're prefixing with a space character:
options = ['2', '3', '7']

choose(text, options)

What if options encode into multiple tokens? In this case we could either use the mean of all tokens for the option or multiply sucessive generated token probabilities.

See the ../prompting/winogrande notebook for an example.